In [1]:
import torch
ckpt = torch.load(
    "/home/pavel/ITMO/NIR2/data/exps_logs/last_concreto_SemanticKITTI2/semseg-ptv3-large-v1m1-kitti-4a-lin2/model/model_best.pth",
    map_location="cpu",  # или "cuda",
    weights_only=False,
)

In [2]:
ckpt['state_dict'].keys()

odict_keys(['seg_head.weight', 'seg_head.bias', 'backbone.embedding.stem.linear.weight', 'backbone.embedding.stem.linear.bias', 'backbone.embedding.stem.norm.weight', 'backbone.embedding.stem.norm.bias', 'backbone.enc.enc0.block0.cpe.0.weight', 'backbone.enc.enc0.block0.cpe.0.bias', 'backbone.enc.enc0.block0.cpe.1.weight', 'backbone.enc.enc0.block0.cpe.1.bias', 'backbone.enc.enc0.block0.cpe.2.weight', 'backbone.enc.enc0.block0.cpe.2.bias', 'backbone.enc.enc0.block0.norm1.0.weight', 'backbone.enc.enc0.block0.norm1.0.bias', 'backbone.enc.enc0.block0.attn.qkv.weight', 'backbone.enc.enc0.block0.attn.qkv.bias', 'backbone.enc.enc0.block0.attn.proj.weight', 'backbone.enc.enc0.block0.attn.proj.bias', 'backbone.enc.enc0.block0.norm2.0.weight', 'backbone.enc.enc0.block0.norm2.0.bias', 'backbone.enc.enc0.block0.mlp.0.fc1.weight', 'backbone.enc.enc0.block0.mlp.0.fc1.bias', 'backbone.enc.enc0.block0.mlp.0.fc2.weight', 'backbone.enc.enc0.block0.mlp.0.fc2.bias', 'backbone.enc.enc0.block1.cpe.0.weight

In [1]:
from transformers import SegformerForSemanticSegmentation

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=14,
    ignore_mismatched_sizes=True,
)

/home/pavel/anaconda3/envs/dl_itmo/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b0-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([14]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([14, 256, 1, 1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
import torch

ckpt = torch.load(
    "/home/pavel/ITMO/NIR2/data/exps_logs/BEV_segformer_sensatUrban/checkpoint.pth.tar",
    map_location="cpu",  # или "cuda",
    weights_only=False,
)


In [13]:
model.load_state_dict(ckpt["state_dict"], strict=False)


<All keys matched successfully>

In [2]:
import open3d as o3d
import numpy as np

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [8]:
import trimesh

path = "/home/pavel/ITMO/NIR2/data/mend_line5_full.glb"
scene = trimesh.load(path, force="scene")
print(scene)
print("Geometry keys:", scene.geometry.keys())
print("Dump meshes count:", len(scene.dump(concatenate=False)))


<trimesh.Scene(len(geometry)=1)>
Geometry keys: odict_keys(['mLineFull2.ply'])
Dump meshes count: 1


In [10]:
import trimesh

path = "/home/pavel/ITMO/NIR2/data/mend_line5_full.glb"
scene = trimesh.load(path, force="scene")

for name, geom in scene.geometry.items():
    print("\n=== Geometry:", name, "===")
    print("Type:", type(geom))
    print("Metadata:", geom.metadata)
    print("Has vertices:", hasattr(geom, "vertices"))
    if hasattr(geom, "vertices"):
        print("Vertices:", len(geom.vertices))
        print("Faces:", len(geom.faces) if hasattr(geom, "faces") else "no faces")



=== Geometry: mLineFull2.ply ===
Type: <class 'trimesh.points.PointCloud'>
Metadata: {'units': 'meters', 'from_gltf_primitive': False}
Has vertices: True
Vertices: 11210449
Faces: no faces


In [11]:
import trimesh
from pathlib import Path

in_file = Path("/home/pavel/ITMO/NIR2/data/mend_line5_full.glb")
out_file = Path("/home/pavel/ITMO/NIR2/data/mend_line5_full.ply")

scene = trimesh.load(in_file, force='scene')
pc = list(scene.geometry.values())[0]   # PointCloud

print("Vertices:", pc.vertices.shape)
if hasattr(pc, "colors") and pc.colors is not None:
    print("Colors:", pc.colors.shape)

pc.export(out_file.as_posix())
print("Saved:", out_file)


Vertices: (11210449, 3)
Colors: (11210449, 4)
Saved: /home/pavel/ITMO/NIR2/data/mend_line5_full.ply


In [6]:
import numpy as np
import open3d as o3d

bin_path = "/home/pavel/ITMO/NIR2/data/SemanticKiti/dataset/dataset/sequences/00/velodyne/000000.bin"

# SemanticKITTI: float32, (x, y, z, intensity)
pts = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 4)
xyz = pts[:, :3]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)

print("Количество точек:", xyz.shape[0])
o3d.visualization.draw_geometries([pcd])


Количество точек: 124668


In [7]:
import trimesh
import numpy as np

glb_path = "/home/pavel/ITMO/NIR2/data/RS10.glb"
scene = trimesh.load(glb_path, force="scene")

print("Geometry keys:", list(scene.geometry.keys()))

for name, geom in scene.geometry.items():
    print("\n---", name, "---")
    if hasattr(geom, "vertices"):
        print("vertices:", geom.vertices.shape)

    # цвет
    if hasattr(geom, "visual") and geom.visual is not None:
        vc = getattr(geom.visual, "vertex_colors", None)
        if vc is not None:
            print("vertex_colors:", np.asarray(vc).shape)

    # иногда данные в metadata / extras
    print("metadata keys:", list(getattr(geom, "metadata", {}).keys()))


Geometry keys: ['RS10_sample(0.05).ply']

--- RS10_sample(0.05).ply ---
vertices: (4352320, 3)
vertex_colors: (4352320, 4)
metadata keys: ['units', 'from_gltf_primitive']


In [8]:
import trimesh
import numpy as np

glb_path = "/home/pavel/ITMO/NIR2/data/RS10.glb"
scene = trimesh.load(glb_path, force="scene")

geom = next(iter(scene.geometry.values()))
xyz = np.asarray(geom.vertices, dtype=np.float32)

rgba = np.asarray(geom.visual.vertex_colors)  # (N,4)
rgb = rgba[:, :3].astype(np.float32) / 255.0  # -> [0,1]

# grayscale luminance
intensity = (0.299 * rgb[:, 0] + 0.587 * rgb[:, 1] + 0.114 * rgb[:, 2]).astype(np.float32)

# SemanticKITTI-like points: (x,y,z,intensity)
pts4 = np.concatenate([xyz, intensity[:, None]], axis=1).astype(np.float32)

out_bin = "/home/pavel/ITMO/NIR2/data/RS10_intens.glb"
pts4.tofile(out_bin)
print("saved:", out_bin, pts4.shape)


saved: /home/pavel/ITMO/NIR2/data/RS10_intens.glb (4352320, 4)


In [9]:
import open3d as o3d
import numpy as np

# загрузка того, что ты сохранил
pts4 = np.fromfile('/home/pavel/ITMO/NIR2/data/RS10_intens.glb', dtype=np.float32).reshape(-1, 4)
xyz = pts4[:, :3]
intensity = pts4[:, 3]

# нормализация (обязательно!)
i = intensity
i = (i - i.min()) / (i.max() - i.min() + 1e-12)

colors = np.stack([i, i, i], axis=1)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)
pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([pcd])


In [10]:
import numpy as np
import open3d as o3d
import itertools

def apply_perm_flip(coord, perm, flips):
    # perm: tuple of indices, e.g. (0,1,2) or (1,2,0)
    c = coord[:, perm].copy()
    c *= np.array(flips, dtype=np.float32)[None, :]
    return c

def score_ground_horizontal(coord, dist=0.2, iters=1000):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(coord)
    plane, inliers = pcd.segment_plane(distance_threshold=dist, ransac_n=3, num_iterations=iters)
    a,b,c,d = plane
    n = np.array([a,b,c], dtype=np.float64)
    n /= (np.linalg.norm(n) + 1e-12)
    # хотим n ~ (0,0,1) или (0,0,-1) => abs(n_z) близко к 1
    return abs(n[2]), plane, len(inliers)

def find_best_axis_mapping(coord, dist=0.2):
    best = None
    perms = list(itertools.permutations([0,1,2], 3))      # 6
    flips = list(itertools.product([1,-1], repeat=3))    # 8

    # чуть центрируем, чтобы RANSAC был стабильнее
    base = coord - coord.mean(0, keepdims=True)

    for p in perms:
        for f in flips:
            c = apply_perm_flip(base, p, f)
            try:
                nz, plane, ninl = score_ground_horizontal(c, dist=dist)
            except Exception:
                continue
            # критерий: максимально горизонтальная плоскость + достаточно inliers
            key = (nz, ninl)
            if best is None or key > best["key"]:
                best = {"perm": p, "flips": f, "key": key, "plane": plane, "ninl": ninl}
    return best

# usage:

pcd = o3d.io.read_point_cloud("/home/pavel/ITMO/NIR2/data/new_tiles/RS10/tile_019.ply")
points = np.asarray(pcd.points)

best = find_best_axis_mapping(points, dist=0.2)
print(best)
coord = apply_perm_flip(points, best["perm"], best["flips"])


{'perm': (0, 2, 1), 'flips': (-1, 1, -1), 'key': (0.9983293893253424, 34684), 'plane': array([ 0.00878589, -0.05710726,  0.99832939,  0.25180485]), 'ninl': 34684}


In [17]:
import numpy as np
import open3d as o3d

PLY_PATH = "/home/pavel/ITMO/NIR2/data/new_tiles/RS10/tile_019.ply"

# ---- 1) axis fix from your auto-search ----
# perm=(0,2,1), flips=(-1,1,-1) => [x,y,z] = [-x0, z0, -y0]
def fix_axes(coord: np.ndarray) -> np.ndarray:
    c = coord[:, [0, 2, 1]].astype(np.float32)
    c[:, 0] *= -1.0
    c[:, 2] *= -1.0
    return c

# ---- 2) choose center strategy ----
def center_points(coord: np.ndarray, mode: str) -> np.ndarray:
    """
    mode:
      - 'none'      : no centering
      - 'mean'      : subtract mean xyz
      - 'bbox'      : subtract bbox center
      - 'ground'    : subtract XY center + set ground median to z=0 (requires plane inliers)
    """
    if mode == "none":
        return coord

    if mode == "mean":
        return coord - coord.mean(axis=0, keepdims=True)

    if mode == "bbox":
        mn = coord.min(axis=0)
        mx = coord.max(axis=0)
        center = (mn + mx) / 2.0
        return coord - center[None, :]

    raise ValueError(f"Unknown center mode: {mode}")

def align_ground_to_xy(coord: np.ndarray, dist_thresh=0.3, num_iter=2000):
    """
    Fits dominant plane and rotates cloud so plane normal aligns with +Z.
    Also shifts z so plane median is at z=0.
    """
    pcd_small = o3d.geometry.PointCloud()
    pcd_small.points = o3d.utility.Vector3dVector(coord)

    plane_model, inliers = pcd_small.segment_plane(
        distance_threshold=dist_thresh,
        ransac_n=3,
        num_iterations=num_iter,
    )
    a, b, c, d = plane_model
    n = np.array([a, b, c], dtype=np.float64)
    n = n / (np.linalg.norm(n) + 1e-12)

    z = np.array([0.0, 0.0, 1.0], dtype=np.float64)
    v = np.cross(n, z)
    s = np.linalg.norm(v)
    c_ = float(np.dot(n, z))

    if s < 1e-8:
        R = np.eye(3, dtype=np.float64)
    else:
        vx = np.array([[0, -v[2], v[1]],
                       [v[2], 0, -v[0]],
                       [-v[1], v[0], 0]], dtype=np.float64)
        R = np.eye(3) + vx + (vx @ vx) * ((1 - c_) / (s * s + 1e-12))

    coord_rot = (R @ coord.T).T.astype(np.float32)

    # shift ground to z=0
    ground = coord_rot[np.array(inliers, dtype=np.int64)]
    if ground.shape[0] > 0:
        z0 = float(np.median(ground[:, 2]))
        coord_rot[:, 2] -= z0

    return coord_rot, plane_model, inliers

def make_axis_frame(size=5.0):
    return o3d.geometry.TriangleMesh.create_coordinate_frame(size=size, origin=[0,0,0])

def make_origin_sphere(radius=0.4):
    sph = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    sph.translate([0,0,0])
    sph.paint_uniform_color([1.0, 0.2, 0.2])
    return sph

def voxel_downsample(coord: np.ndarray, colors=None, voxel=0.15):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(coord)
    if colors is not None:
        pcd.colors = o3d.utility.Vector3dVector(colors)
    pcd2 = pcd.voxel_down_sample(voxel_size=voxel)
    c2 = np.asarray(pcd2.points, dtype=np.float32)
    col2 = np.asarray(pcd2.colors, dtype=np.float32) if colors is not None else None
    return c2, col2

def save_ply(path, coord, color=None):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(coord.astype(np.float32))

    if color is not None:
        col = color.astype(np.float32)
        if col.max() > 1.0:
            col = col / 255.0
        pcd.colors = o3d.utility.Vector3dVector(col)

    o3d.io.write_point_cloud(path, pcd)
    print(f"[OK] Saved transformed ply: {path}")
    
def main(center_mode="mean", do_ground_align=True):
    pcd = o3d.io.read_point_cloud(PLY_PATH)
    coord = np.asarray(pcd.points, dtype=np.float32)
    color = np.asarray(pcd.colors, dtype=np.float32)
    if color.size == 0:
        color = None

    # finite filter
    m = np.isfinite(coord).all(axis=1)
    coord = coord[m]
    if color is not None:
        color = color[m]

    # (A) fix axes
    coord = fix_axes(coord)

    # (B) optional: downsample for fast viz & stable plane fit
    coord_ds, color_ds = voxel_downsample(coord, color, voxel=0.2)

    # (C) choose center
    coord_ds = center_points(coord_ds, center_mode)
    print(coord_ds)

    # (D) optional ground align + shift z to 0
    if do_ground_align:
        coord_ds, plane, inliers = align_ground_to_xy(coord_ds, dist_thresh=0.3)
        print("plane:", plane, "inliers:", len(inliers))
    else:
        plane, inliers = None, None

    # build pcd for viz
    pcd_v = o3d.geometry.PointCloud()
    pcd_v.points = o3d.utility.Vector3dVector(coord_ds)
    if color_ds is not None and color_ds.size > 0:
        pcd_v.colors = o3d.utility.Vector3dVector(color_ds)

    # axes + origin marker
    axis = make_axis_frame(size=5.0)
    origin = make_origin_sphere(radius=0.4)

    o3d.visualization.draw_geometries([pcd_v, axis, origin])
    out_ply = PLY_PATH.replace(".ply", "_fixed_axes_center_ground.ply")
    save_ply(out_ply, coord_ds, color_ds)



    
# if __name__ == "__main__":
    # Попробуй варианты:
    # center_mode: 'none' | 'mean' | 'bbox'
    # do_ground_align: True/False
main(center_mode="bbox", do_ground_align=True)


[[-6.9521484   3.9453125  -0.68322754]
 [-6.9658203   3.7841797  -0.69241333]
 [-6.435547    3.9482422  -0.70207214]
 ...
 [-0.8183594  -0.41601562 -1.0646362 ]
 [ 5.9746094   1.6123047  -0.8948517 ]
 [ 5.9814453   1.4042969  -0.9435425 ]]
plane: [ 0.0057095  -0.05254462  0.99860226  0.9098029 ] inliers: 3618
[OK] Saved transformed ply: /home/pavel/ITMO/NIR2/data/new_tiles/RS10/tile_019_fixed_axes_center_ground.ply


In [22]:
import open3d as o3d
import numpy as np

# Загружаем point cloud
pcd = o3d.io.read_point_cloud("/home/pavel/ITMO/NIR2/data/out/pred.ply")
# pcd = o3d.io.read_point_cloud("/home/pavel/ITMO/NIR2/data/tiles/sensatUrban/birmingham_block_8/ply/birmingham_block_8_ix000_iy000_n1174720.ply")

# Число точек
points = np.asarray(pcd.points)
print("Количество точек:", points.shape[0])
print(pcd.has_colors())
print(pcd)
# Визуализация
o3d.visualization.draw_geometries([pcd])


# 19, 22, 25

Количество точек: 4961
True
PointCloud with 4961 points.


In [15]:
import plyfile
from plyfile import PlyData

ply = PlyData.read("/home/pavel/ITMO/NIR2/data/mend_line5_full.ply")
print(ply)


ply
format binary_little_endian 1.0
comment https://github.com/mikedh/trimesh
element vertex 11210449
property float x
property float y
property float z
property uchar red
property uchar green
property uchar blue
property uchar alpha
end_header


In [16]:
import numpy as np
import open3d as o3d
from plyfile import PlyData

path = "/home/pavel/ITMO/NIR2/data/mend_line5_full.ply"

print("\n=== ОТЧЁТ О ЦВЕТЕ В PLY ===")

# --- 1. Анализ через plyfile (без магии Open3D) ---
ply = PlyData.read(path)
vertex = ply["vertex"]

has_r = "red" in vertex.data.dtype.names
has_g = "green" in vertex.data.dtype.names
has_b = "blue" in vertex.data.dtype.names
has_any_color = has_r or has_g or has_b

print(f"Файл содержит цветовые поля R,G,B: {has_any_color}")
if has_any_color:
    print(f"- Поля цвета: {[c for c in ['red','green','blue'] if c in vertex.data.dtype.names]}")

num_points_file = len(vertex.data)
print(f"Точек в файле: {num_points_file:,}")

# --- 2. Анализ через Open3D (который может интерполировать) ---
pcd = o3d.io.read_point_cloud(path)
points = np.asarray(pcd.points)
colors = np.asarray(pcd.colors)

print(f"\nOpen3D считает:")
print(f"- Кол-во точек: {points.shape[0]:,}")
print(f"- Кол-во цветов: {colors.shape[0]:,}")
print(f"- has_colors(): {pcd.has_colors()}")

# --- 3. Настоящий процент наличия цвета ---
if has_any_color:
    # количество точек с color в файле = число точек, у которых есть r,g,b
    num_color_points = num_points_file  # если есть поля, они для всех
    perc = 100.0
else:
    # вообще нет цветовых полей
    num_color_points = 0
    perc = 0.0

print(f"\nРЕАЛЬНЫЙ процент точек с цветом: {perc:.2f}%")

# --- 4. Доп инфо о цветах ---
if colors.size > 0:
    print("\nДиапазон цветов (Open3D интерпретации):")
    print("min:", colors.min(axis=0))
    print("max:", colors.max(axis=0))

print("\n=== Конец отчёта ===")



=== ОТЧЁТ О ЦВЕТЕ В PLY ===
Файл содержит цветовые поля R,G,B: True
- Поля цвета: ['red', 'green', 'blue']
Точек в файле: 11,210,449

Open3D считает:
- Кол-во точек: 11,210,449
- Кол-во цветов: 11,210,449
- has_colors(): True

РЕАЛЬНЫЙ процент точек с цветом: 100.00%

Диапазон цветов (Open3D интерпретации):
min: [0. 0. 0.]
max: [1. 1. 1.]

=== Конец отчёта ===


In [27]:
import cv2

# ====== CONFIG ======
MASK_PATH = "/home/pavel/ITMO/NIR2/Otchet/rank09_miou81.26_b0071_i01_pred_color.png"          # вход: RGB (Cityscapes color mask)
OUT_PATH  = "/home/pavel/ITMO/NIR2/Otchet/rank09_miou81.26_b0071_i01_pred_color_vector_overlay.png"

# метры/пиксель тут не нужны, мы рисуем в пикселях (как чертеж на картинке)
# если потом будешь экспортировать в DXF/GeoJSON — тогда пригодится resolution.

CITYSCAPES_COLORS_19 = {
    0:  (128, 64, 128),   # road
    1:  (244, 35, 232),   # sidewalk
    2:  (70, 70, 70),     # building
    # 3:  (102, 102, 156),  # wall
    # 4:  (190, 153, 153),  # fence
    # 5:  (153, 153, 153),  # pole
    # 6:  (250, 170, 30),   # traffic light
    # 7:  (220, 220, 0),    # traffic sign
    # 8:  (107, 142, 35),   # vegetation
    # 9:  (152, 251, 152),  # terrain
    # 10: (70, 130, 180),   # sky
    # 11: (220, 20, 60),    # person
    # 12: (255, 0, 0),      # rider
    # 13: (0, 0, 142),      # car
    # 14: (0, 0, 70),       # truck
    # 15: (0, 60, 100),     # bus
    # 16: (0, 80, 100),     # train
    # 17: (0, 0, 230),      # motorcycle
    # 18: (119, 11, 32),    # bicycle
}

# какие классы рисуем как "линии чертежа"
# (BGR) — потому что OpenCV рисует в BGR
CLASSES_TO_DRAW = [
    # ("road",       0,  (0, 0, 0),       2),  # черные линии
    ("building",   2,  (0, 0, 0),       2),
    # ("vegetation", 8,  (0, 120, 0),     2),  # зеленые линии
    # ("terrain",    9,  (60, 160, 60),   2),  # светло-зеленые
    # можно добавить тротуар:
    # ("sidewalk",   1,  (80, 80, 80),    2),
]

# морфология (подстрой под свои маски)
MORPH = {
    "road":       dict(close_ks=11, open_ks=7),
    "building":   dict(close_ks=7,  open_ks=3),
    "vegetation": dict(close_ks=7,  open_ks=5),
    "terrain":    dict(close_ks=7,  open_ks=5),
    "sidewalk":   dict(close_ks=9,  open_ks=5),
}

# упрощение контуров (epsilon в пикселях): больше -> более "чертежно"
APPROX_EPS = {
    "road": 2.0,
    "building": 1.5,
    "vegetation": 2.5,
    "terrain": 2.5,
    "sidewalk": 2.0,
}

# фильтры по площади (в пикселях): мелкие куски убираем
MIN_AREA_PX = {
    "road": 500,
    "building": 200,
    "vegetation": 150,
    "terrain": 150,
    "sidewalk": 200,
}

# ====== FUNCTIONS ======
def rgb_to_binary(rgb_mask, color_rgb):
    """rgb_mask: (H,W,3) RGB uint8 ; returns (H,W) uint8 {0,1}"""
    c = np.array(color_rgb, dtype=np.uint8)
    return np.all(rgb_mask == c, axis=-1).astype(np.uint8)

def clean_binary(binary01, close_ks=7, open_ks=5, iters=1):
    """binary01: (H,W) {0,1} -> cleaned {0,1}"""
    img = (binary01 > 0).astype(np.uint8) * 255
    if close_ks and close_ks > 1:
        k = np.ones((close_ks, close_ks), np.uint8)
        img = cv2.morphologyEx(img, cv2.MORPH_CLOSE, k, iterations=iters)
    if open_ks and open_ks > 1:
        k = np.ones((open_ks, open_ks), np.uint8)
        img = cv2.morphologyEx(img, cv2.MORPH_OPEN, k, iterations=iters)
    return (img > 0).astype(np.uint8)

def extract_contours(binary01):
    """returns list of contours"""
    img = (binary01 > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    return contours

def contour_area_px(cnt):
    return float(cv2.contourArea(cnt))

def approx_contour(cnt, eps_px):
    return cv2.approxPolyDP(cnt, epsilon=float(eps_px), closed=True)

def draw_contours(canvas_bgr, contours, color_bgr, thickness=2, eps_px=2.0, min_area_px=100):
    """
    Рисуем контуры как полилинии. Можно рисовать и исходные, и упрощенные.
    """
    for cnt in contours:
        if contour_area_px(cnt) < float(min_area_px):
            continue
        ac = approx_contour(cnt, eps_px=eps_px)
        # cv2.polylines ожидает shape (N,1,2)
        cv2.polylines(canvas_bgr, [ac], isClosed=True, color=color_bgr, thickness=int(thickness))

# ====== MAIN ======
def main():
    # читаем маску (OpenCV читает BGR) -> переводим в RGB
    bgr = cv2.imread(MASK_PATH, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Can't read {MASK_PATH}")
    rgb = bgr[:, :, ::-1].copy()

    H, W = rgb.shape[:2]
    # белый холст (чертеж)
    canvas = np.full((H, W, 3), 255, dtype=np.uint8)

    # если хочешь — можно подложить полупрозрачную исходную маску:
    # canvas = (0.8*canvas + 0.2*bgr).astype(np.uint8)

    for name, cls_id, color_bgr, thickness in CLASSES_TO_DRAW:
        color_rgb = CITYSCAPES_COLORS_19[cls_id]

        bin01 = rgb_to_binary(rgb, color_rgb)

        m = MORPH.get(name, dict(close_ks=7, open_ks=5))
        bin01 = clean_binary(bin01, close_ks=m["close_ks"], open_ks=m["open_ks"], iters=1)

        contours = extract_contours(bin01)

        eps = APPROX_EPS.get(name, 2.0)
        min_area = MIN_AREA_PX.get(name, 150)

        draw_contours(
            canvas,
            contours,
            color_bgr=color_bgr,
            thickness=thickness,
            eps_px=eps,
            min_area_px=min_area,
        )

        # подпись класса (опционально): можно закомментить
        # cv2.putText(canvas, name, (10, 30 + 25*i), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_bgr, 2)

    cv2.imwrite(OUT_PATH, canvas)
    print("Saved:", OUT_PATH)

# запуск
main()


Saved: /home/pavel/ITMO/NIR2/Otchet/rank09_miou81.26_b0071_i01_pred_color_vector_overlay.png
